# 66 · PT pathways: F-statistic ranking against clustering-first, on one gene model

**The question.** Two pipelines look for pathways whose expression differs between two groups along
the proximal tubule (PT). Do they find the same pathways when everything except the method is the
same?

- **Method A, F-statistic ranking (ours).** Every gene gets one statistic, `T_spatial`: how much
  the group difference changes along the PT beyond a constant offset. All genes are ranked by it,
  and each pathway is tested on that ranking with a standard package: limma's **cameraPR**
  (primary) and gseapy's preranked **GSEA** (beside it).
- **Method B, clustering-first (Caleb Hallinan's `pseudospace_reconstruction`).** Genes are first
  selected by replication rules, then sorted by the shape of their difference, clustered, and the
  gene groups and clusters are named by over-representation analysis (ORA).

**What is held equal.**
- **One gene model for both:** a negative-binomial (NB) GLM on raw counts with a 6-df spline
  (`pseudospace.pathway_pipelines.standard_gene_model`). Method A takes its `T_spatial`; method B
  takes its per-specimen curves in place of its own count model.
- The same structures, coordinate, grid, gene universe, gene sets (10–300 genes), zones and
  specimen relabelings. Section 4 lists every change from Caleb's defaults.

**Two datasets, the same inputs for both methods:**

| Dataset | Structures | Genes | Pathways | Reference | Case | Coordinate |
|---|---|---|---|---|---|---|
| Human vs mouse PT | 12,272 | 11,071 | 1,513 | 2 control mice | 2 cortex sections of 1 human donor | SCF13 (notebook 13's scFates) |
| AKI vs control mouse PT | 16,991 | 9,107 | 1,429 | 2 control mice | 2 AKI mice | mouse-only PT DPT (mouse_only_v5), as notebook 45's positive control |

**How every result is judged.** Each dataset is analysed three times: the **condition split**
(case against reference) and the two **balanced relabelings**, which pair one case specimen with
one reference specimen. A relabeling has no condition difference left to find, so its calls
measure specimen-to-specimen noise.

**Sections:**
1. Setup, inputs and the redundancy rule.
2. The shared gene model.
3. Method A: `T_spatial` ranking with cameraPR, GSEA and our joint test for reference.
4. Method B: Caleb's stages 06–07 on the shared curves, and the standardization table.
5. Comparison: method tables, overlap, what each finds alone, worked examples.
6. Summary.
7. Run record.

**Assumptions.** Each coordinate is treated as exact. Specimens, not structures, are the
replicates. With two specimens per group every result is descriptive.

## 1 · Setup

In [ ]:
import argparse
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
upstream13 = results_root / 'minimal_pt_scfates'
mouse_v5 = results_root / 'mouse_only_v5'
universe_file = results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv'      # notebook 45's universe
nb45_cache = results_root / 'pt_revision_method' / 'stage_cache'                            # read only, guards
nb65_genes = results_root / 'pt_gene_model_comparison' / 'tables' / 'gene_statistics_gaussian_and_nb.csv'  # read only, guard
output = results_root / 'pt_pathway_pipelines_compared'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [coordinate_file, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            universe_file, mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad',
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 13 and 36 and the mouse_only_v5 workflow first)')

NOTEBOOK_LOGIC_VERSION = '66.pipelines.1'   # Bump when a cached calculation changes.
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05      # discovery level and notebook 31's relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                    # notebook 61's criterion 2: share of relabeled p <= 0.05
CAMERA_COR = .01                                 # limma's default inter.gene.cor; the relabelings imply 0.0096
GSEA_PERMUTATIONS, GSEA_SEED = 1000, 12
N_NULL, SEED = 9999, 12                          # matched null of the reference joint test (notebook 37)
BASIS_DF, GRID_POINTS = 6, 101                   # the shared gene model
ABSENT = .01                                     # nearly absent: detected in < 1% of one group's structures
SET_SIZES = (10, 300)
REDUNDANCY_JACCARD = .5                          # fixed before any result (section 1.2)
ZONE_MARKERS = {'PT-S1': ['Slc5a12'], 'PT-S2': ['Slc13a3'], 'PT-S3': ['Slc16a9']}   # Caleb's shipped mouse kidney config
ZONE_SHORT = {'PT-S1': 'S1', 'PT-S2': 'S2', 'PT-S3': 'S3'}
NODE_SPACING = 1 / 29                            # SCF13 is a 30-node scFates curve; border tolerance = half of it
EXTERNAL_COMMIT = '59da04b'                      # pseudospace_reconstruction, as installed
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '36'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import distribution, version
from IPython.display import display

direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown'), 'url': direct_url.get('url', '')}
assert EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), f'pseudospace_reconstruction is not commit {EXTERNAL_COMMIT}: {EXTERNAL}'

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
A_COLOR, B_COLOR, REF_COLOR = '#2a78d6', '#eb6834', '#8a8984'   # method A, method B, our reference test
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS, '· pseudospace_reconstruction', EXTERNAL['version'],
      EXTERNAL['commit'][:7])

### 1.1 · Inputs: two datasets

**Human vs mouse PT.** Notebook 37's construction on SCF13 (`pseudospace.coordinate_inputs.pt_inputs`):
- the structures inside every specimen's 1st–99th percentile range of the coordinate;
- the genes measured in both species and detected in ≥ 2% of structures (equal-specimen mean);
- the pathways with 10–300 tested genes.

**AKI vs control mouse PT.** Exactly the inputs of notebook 45's AKI positive control (its cells 4, 8
and 10, mirrored here because they are not saved as a table):
- mouse_only_v5's PT structures with an S1, S2 or S3 label, inside every specimen's 1st–99th
  percentile range of the mouse-only PT DPT coordinate, rescaled to [0, 1];
- the genes of notebook 12's universe that are measured in this pool and detected in ≥ 2% of
  structures (mean over the four mice);
- notebook 37's pathways restricted to those genes, 10–300 members: 1,429 pathways.

In both, the library size is the total count over all orthologs measured in the dataset, and the
NB offset is its log. A guard checks the AKI genes, pathways and matching strata against notebook
45's cached pool.

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_pipelines import partition_groups
from pseudospace.pathway_remodeling import matching_strata
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
hm = pt_inputs(results_root, data_root, read_coordinate(coordinate_file))
assert len(hm['position']) == 12272 and len(hm['genes']) == 11071 and len(hm['gene_sets']) == 1513, "notebook 37's universe changed"
DATA = {'human_vs_mouse': {
    'title': 'Human vs mouse PT', 'case_name': 'human', 'reference_name': 'mouse', 'coordinate': 'SCF13',
    'counts': hm['counts'].tocsr(), 'library': hm['library'], 'position': hm['position'],
    'specimen': hm['specimen'].astype(str), 'segment': pd.Series(hm['segment']).str.replace('PT-', '', regex=False).to_numpy(),
    'case': hm['human'].astype(bool), 'genes': list(hm['genes']), 'sets': hm['gene_sets'], 'strata': hm['strata'],
    'lognorm': hm['Y'].tocsr()}}
del hm

# AKI vs control: notebook 45 cells 4, 8 and 10 (build_pool for one source of four mice)
universe_genes = pd.read_csv(universe_file, index_col=0).index.tolist()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe_genes,
    ortholog_map=orthologs, library_name=lib, tested=universe_genes, min_genes=1) for lib in LIBRARIES], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(SET_SIZES[0], min(SET_SIZES[1], len(universe_genes) - 1))]
gene_sets_all = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
assert len(gene_sets_all) == 1513, "notebook 37's pathways changed"

v5 = ad.read_h5ad(mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad')
v5 = v5[v5.obs.broad_tubule_marker_call.astype(str).eq('PT').to_numpy()
        & v5.obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()].copy()
support = v5.obs.groupby('sample', observed=True).pt_subset_scanpy_dpt.quantile([.01, .99]).unstack()
coordinate = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
v5 = v5[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
aki_position = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
aki_position = (aki_position - aki_position.min()) / (aki_position.max() - aki_position.min())
panel = np.isin(v5.var_names, list(set(orthologs.mouse_symbol)))
aki_raw = sparse.csr_matrix(v5.layers['counts'])[:, panel]
aki_var = v5.var_names[panel]
aki_specimen = v5.obs['sample'].astype(str).to_numpy()
aki_segment = v5.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
del v5
measured = sorted(set(aki_var[np.asarray(aki_raw.sum(axis=0)).ravel() > 0]))
aki_counts_all = sparse.csr_matrix(aki_raw[:, pd.Index(aki_var).get_indexer(measured)], dtype=float)
aki_library = np.asarray(aki_counts_all.sum(axis=1)).ravel()
aki_lognorm_all = aki_counts_all.multiply((1e4 / aki_library)[:, None]).tocsr()
aki_lognorm_all.data = np.log1p(aki_lognorm_all.data)
pool_genes = [g for g in universe_genes if g in set(measured)]
columns = pd.Index(measured).get_indexer(pool_genes)
x = aki_lognorm_all[:, columns]
per_donor = {}
for name in np.unique(aki_specimen):
    rows, seg = x[aki_specimen == name], aki_segment[aki_specimen == name]
    present = np.array([np.asarray((rows[seg == s] > 0).sum(axis=0)).ravel() > 0 for s in np.unique(seg)])
    per_donor[name] = (np.asarray(rows.mean(axis=0)).ravel(), np.asarray((rows > 0).mean(axis=0)).ravel(), present.mean(axis=0))
aki_covariates = pd.DataFrame({key: np.mean([per_donor[d][k] for d in per_donor], axis=0)
                               for k, key in enumerate(('mean_expression', 'detection', 'coverage'))}, index=pool_genes)
aki_genes = list(aki_covariates.index[aki_covariates.detection.ge(.02)])
keep = set(aki_genes)
aki_sets = {p: [g for g in v if g in keep] for p, v in gene_sets_all.items()}
aki_sets = {p: v for p, v in aki_sets.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(keep) - 1)}
aki_columns = pd.Index(measured).get_indexer(aki_genes)
DATA['aki_vs_control'] = {
    'title': 'AKI vs control mouse PT', 'case_name': 'AKI', 'reference_name': 'control', 'coordinate': 'mouse-only PT DPT',
    'counts': aki_counts_all[:, aki_columns].tocsr(), 'library': aki_library, 'position': aki_position,
    'specimen': aki_specimen, 'segment': aki_segment, 'case': np.isin(aki_specimen, ['IR2A2', 'IR2A4']),
    'genes': aki_genes, 'sets': aki_sets, 'strata': matching_strata(aki_covariates.loc[aki_genes], bins=3),
    'lognorm': aki_lognorm_all[:, aki_columns].tocsr()}
del aki_raw, aki_counts_all, aki_lognorm_all, x
assert len(aki_position) == 16991 and len(aki_genes) == 9107 and len(aki_sets) == 1429, "notebook 45's AKI inputs changed"
pools45 = sorted(nb45_cache.glob('pool_aki__*.npz'), key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
if pools45:
    with np.load(pools45[-1]) as stored:
        assert list(stored['genes'].astype(str)) == aki_genes and list(stored['pathways'].astype(str)) == list(aki_sets)
        assert np.array_equal(stored['strata'], DATA['aki_vs_control']['strata'])
    print("AKI genes, pathways and strata = notebook 45's cached pool")

DATASETS = list(DATA)
PARTITIONS, SHORT = {}, {}


def group_one(ds, label):
    """The specimens in group 1 of a partition (the case specimens for the condition split)."""
    d = DATA[ds]
    return sorted(set(d['specimen'][PARTITIONS[ds][label][0] == 1]))


for ds, d in DATA.items():
    assert d['counts'].shape == d['lognorm'].shape == (len(d['position']), len(d['genes']))
    assert all(set(v) <= set(d['genes']) for v in d['sets'].values())
    PARTITIONS[ds] = partition_groups(d['specimen'], d['case'])
    labels = list(PARTITIONS[ds])
    SHORT[ds] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    detected = {side: np.asarray((d['counts'][mask] > 0).mean(axis=0)).ravel()
                for side, mask in (('case', d['case']), ('reference', ~d['case']))}
    d['detection_case'], d['detection_reference'] = detected['case'], detected['reference']
    d['nearly_absent'] = np.minimum(detected['case'], detected['reference']) < ABSENT
    d['mean_expression'] = np.mean([np.asarray(d['lognorm'][d['specimen'] == s].mean(axis=0)).ravel()
                                    for s in sorted(set(d['specimen']))], axis=0)     # equal-specimen mean log1p CP10k
    d['input_key'] = {'counts': digest(d['counts']), 'library': d['library'], 'position': d['position'],
                      'specimen': d['specimen'], 'genes': d['genes']}

INPUT_TABLE = pd.DataFrame({d['title']: {
    'structures': len(d['position']), 'genes': len(d['genes']), 'pathways': len(d['sets']),
    'reference specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in sorted(set(d['specimen'][~d['case']]))),
    'case specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in sorted(set(d['specimen'][d['case']]))),
    **{f'{SHORT[ds][label]} (one side)': ' + '.join(group_one(ds, label)) for label in list(PARTITIONS[ds])[1:]},
    'genes nearly absent in one group': f"{int(d['nearly_absent'].sum()):,} ({d['nearly_absent'].mean():.1%})"}
    for ds, d in DATA.items()})
INPUT_TABLE.to_csv(tables / 'inputs.csv')
display(INPUT_TABLE)
print(f'Inputs {time.perf_counter() - t0:.0f} s')

### 1.2 · The redundancy rule, fixed before any result

Reactome holds many near-duplicate pathways, so raw call counts overstate how many distinct
findings a method makes. Overlaps between methods (section 5) are therefore also counted over
**redundancy clusters**. The rule is fixed here, before any method has run:
- the distance between two pathways is 1 − Jaccard of their full membership within the tested genes;
- average linkage, cut where the mean Jaccard between clusters falls below 0.5;
- each cluster is named by its **medoid**, the member with the highest mean Jaccard to the rest
  (ties: the smaller set, then the name).

This is Caleb's own redundancy rule (`compare.pathways.cluster_gene_sets`, his stage 07), and his
function computes it here for both methods. A call set is collapsed to the clusters it touches.

In [ ]:
from pseudospace_reconstruction.compare.pathways import cluster_gene_sets
from pseudospace_reconstruction.io.gene_sets import build_gene_sets

EXT_SETS, REDUNDANCY = {}, {}
for ds, d in DATA.items():
    sets = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in d['sets'].items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, d['genes'], min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
    ids = [f'{lib}::{name}' for lib, name in zip(sets.library, sets.pathway)]
    assert sorted(ids) == sorted(d['sets']), "Caleb's gene-set builder and ours disagree on the tested pathways"
    assert all(set(sets.member_names(i)) == set(d['sets'][p]) for i, p in enumerate(ids)), 'members differ'
    EXT_SETS[ds] = sets
    clusters = cluster_gene_sets(sets.membership(), sets.names, REDUNDANCY_JACCARD)
    clusters.index = pd.Index(ids, name='pathway_id')
    medoid = clusters[clusters.is_representative].reset_index().set_index('cluster_id').pathway_id
    clusters['representative'] = clusters.cluster_id.map(medoid)
    REDUNDANCY[ds] = clusters.reindex(list(d['sets']))
    REDUNDANCY[ds].to_csv(tables / f'redundancy_clusters_{ds}.csv')
print({DATA[ds]['title']: f"{len(DATA[ds]['sets']):,} pathways in {REDUNDANCY[ds].cluster_id.nunique():,} redundancy clusters"
       for ds in DATASETS})


def representatives(ds, calls):
    """The redundancy clusters a call set touches, named by their medoids."""
    return set(REDUNDANCY[ds].representative.reindex(sorted(calls)))


NAMES = {ds: pd.Series({p: p.split('::', 1)[1] for p in DATA[ds]['sets']}).str.replace(r' R-HSA-\d+$', '', regex=True)
         for ds in DATASETS}

## 2 · The shared gene model

**What it is** (`pseudospace.pathway_pipelines.standard_gene_model`, frozen in
`docs/results/pt-gene-model-nb.md`):
- an NB GLM per gene on raw counts, log link, offset log(library);
- a cubic B-spline in position, 6 df plus the intercept, knots at the position quartiles;
- prior weights wᵢ = n / (2 · J · n_j): each group half the weight, each specimen an equal share;
- one dispersion α per gene, estimated under the condition split's full model and reused for both
  relabelings, as notebooks 37 and 65 do;
- in a relabeling, the condition's own smooth shape enters every model as a nuisance term.

**What each method takes from it:**
- **Method A:** `T_spatial`, the NB likelihood ratio of the group × position terms beyond a group
  offset. It is a rank, not a calibrated gene test: its degrees of freedom count structures.
- **Method B:** one curve per specimen (intercept + the same spline, fitted on that specimen's
  structures with the gene's α) on a common 101-point grid. The grid spans the intersection of the
  four specimens' 1st–99th percentile ranges. These curves do not depend on the partition.

**Guards.** The specimen curves are identical in all three partitions. For human vs mouse, the
statistics reproduce notebook 65's NB `T_spatial` for all three partitions.

In [ ]:
from pseudospace import nb_gam, pathway_calibration, pathway_decoys, pathway_pipelines, pathway_remodeling
from pseudospace.pathway_pipelines import GeneModel, standard_gene_model

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in
                                               (nb_gam, pathway_calibration, pathway_decoys, pathway_pipelines, pathway_remodeling)])
SECONDS = {}   # uncached compute time of every cached stage


def cached(stage, compute, *, params=None, inputs=None, key=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=key or code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def slug(label):
    return label.replace(':', '_').replace('+', '_')


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, group_names=model.groups.index.to_numpy().astype(str), group_values=model.groups.to_numpy(),
               alpha=np.asarray(model.dispersion['alpha'], float), alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


MODELS = {}
for ds, d in DATA.items():
    MODELS[ds] = {}
    for label, (group, nuisance) in PARTITIONS[ds].items():
        dispersion = None if label == 'species' else MODELS[ds]['species'].dispersion
        payload = cached(f'gene_model_{ds}_{slug(label)}', lambda d=d, group=group, nuisance=nuisance, dispersion=dispersion:
                         model_payload(standard_gene_model(d['counts'], d['library'], d['position'], group, d['specimen'],
                                                           genes=d['genes'], nuisance_shape=nuisance, dispersion=dispersion,
                                                           basis_df=BASIS_DF, n_jobs=N_JOBS)),
                         params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': label},
                         inputs={**d['input_key'], 'group': group, 'nuisance': nuisance,
                                 'alpha': None if dispersion is None else dispersion['alpha']})
        MODELS[ds][label] = model_from(payload, d['genes'])
    condition = MODELS[ds]['species']
    assert len(condition.grid) == GRID_POINTS
    for label, model in MODELS[ds].items():
        np.testing.assert_array_equal(model.grid, condition.grid)
        for name, curve in model.specimen_curves.items():
            np.testing.assert_allclose(curve, condition.specimen_curves[name], rtol=1e-9, atol=1e-9)

guard_log = []
if nb65_genes.is_file():
    nb65 = pd.read_csv(nb65_genes, index_col=0).reindex(DATA['human_vs_mouse']['genes'])
    if 'T_spatial_nb' in nb65:
        for label, model in MODELS['human_vs_mouse'].items():
            column = 'T_spatial_nb' if label == 'species' else f'T_spatial_nb|{label}'
            np.testing.assert_allclose(model.statistics.T_spatial, nb65[column], rtol=1e-6, atol=1e-6)
        np.testing.assert_allclose(MODELS['human_vs_mouse']['species'].statistics.alpha, nb65.dispersion, rtol=1e-6)
        guard_log.append("human vs mouse: T_spatial of all three partitions and α = notebook 65's NB statistics (rtol 1e-6)")
guard_log.append('specimen curves identical in all three partitions of both datasets')
print('\n'.join('· ' + line for line in guard_log))

MODEL_TABLE = pd.DataFrame([{
    'dataset': DATA[ds]['title'], 'partition': SHORT[ds][label], 'median α': float(np.median(m.statistics.alpha)),
    'non-converged (median-filled)': int((~m.statistics.converged).sum()), 'separated (flagged)': int(m.statistics.separated.sum()),
    'median T_spatial': float(m.statistics.T_spatial.median()), 'grid': f'[{m.grid[0]:.3f}, {m.grid[-1]:.3f}], {len(m.grid)} points',
    'compute seconds': round(SECONDS[f'gene_model_{ds}_{slug(label)}'], 1)}
    for ds in DATASETS for label, m in MODELS[ds].items()])
MODEL_TABLE.to_csv(tables / 'gene_model_summary.csv', index=False)
display(MODEL_TABLE.round(3))

# Gene descriptors used in section 5 (condition split): the mean NB log rate ratio (case / reference) over the grid.
for ds, d in DATA.items():
    m = MODELS[ds]['species']
    d['log_ratio'] = pd.Series(m.delta.mean(axis=1), index=d['genes'])
    d['typical_shift'] = np.median(m.delta, axis=0)      # Caleb's typical-gene reference (natural log), per grid point

## 3 · Method A: F-statistic ranking with standard set tests

**What it does.**
1. Every gene is ranked by `T_spatial` from the shared gene model.
2. The ranking is converted to rank-normal scores, Φ⁻¹((rank − 0.5) / G). The NB likelihood ratio is
   heavy-tailed, and a few extreme genes would otherwise dominate a mean-based set test.
3. Each pathway is tested on these scores with a standard preranked package:
   - **cameraPR (primary):** limma's `cameraPR` with its defaults (`inter.gene.cor = 0.01`,
     `use.ranks = FALSE`). It compares the pathway's mean score with the other genes' in a t-test
     whose variance is inflated for correlation between the pathway's genes. The default 0.01 is
     close to the correlation the relabelings imply for this statistic (ρ ≈ 0.0096, notebook 65).
     Sensitivity: `use.ranks = TRUE`, its rank-sum version.
   - **gseapy GSEA (beside it):** preranked GSEA on the same scores, 1,000 gene-set permutations
     (`pathway_pipelines.rank_first_gsea`), and once on the raw `T_spatial`. Plain GSEA has no
     correlation correction and is known not to be specific here (notebook 61).

**One call means:** the pathway's genes rank higher on `T_spatial` than other genes do: their
group difference changes shape along the PT more than for typical genes. cameraPR calls are
direction "Up" with limma's BH FDR ≤ 0.05.

**It also returns:** every pathway's score, direction and p-value, so pathways can be ranked and
calibration can be checked.

**What it cannot tell you:**
- a constant group offset, which `T_spatial` removes on purpose;
- where along the PT the difference sits, or which way it goes; those come from the gene curves;
- whether a single gene is significant.

**Reference row (not the user's method).** Our own custom joint test, shown for continuity with
notebooks 37 and 65: a rank-AUC against 9,999 random gene sets matched on expression, detection and
coverage, with z divided by √VIF. Here the VIF is fitted on the two relabelings' matched z
(`pathway_decoys.decoy_inflation`, E[z²] = a(1 + (m − 1)ρ)). Because the VIF is fitted on the same
relabelings that judge calibration, its calibration share is not an independent check; a
cross-fitted version (fit on one relabeling, judge the other) is printed beside it.

For calibration, every method's p-value is one-sided for enrichment at the top: cameraPR's p/2
when Up, else 1 − p/2; GSEA's nominal p made one-sided the same way by the sign of NES.

In [ ]:
from joblib import Parallel, delayed, parallel_config
from scipy.stats import norm
from threadpoolctl import threadpool_limits

from pseudospace.pathway_calibration import normal_matched_q
from pseudospace.pathway_decoys import decoy_inflation, joint_matched_test
from pseudospace.pathway_pipelines import camera_pr, rank_first_gsea, rank_normal_scores
from pseudospace.pathway_remodeling import matched_pathway_tests

A_METHODS = {'cameraPR': 'A · cameraPR (primary)', 'cameraPR ranks': 'A · cameraPR, use.ranks = TRUE',
             'GSEA': 'A · GSEA, rank-normal scores', 'GSEA raw': 'A · GSEA, raw T_spatial',
             'joint': 'Reference · NB joint test, relabeling-fitted VIF (ours)'}
TESTS = {ds: {m: {} for m in A_METHODS} for ds in DATASETS}   # TESTS[ds][method][label]: pathway_id, p_one_sided, call, ...
RUNTIME = {ds: {} for ds in DATASETS}


def _matched_one(frame, sets, strata):
    with threadpool_limits(1):
        return matched_pathway_tests(frame, sets, strata, n_null=N_NULL, seed=SEED)


JOINT_FIT = {}
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    swaps = labels[1:]
    seconds = {m: 0. for m in A_METHODS}
    for label, model in MODELS[ds].items():
        score = rank_normal_scores(model.statistics.T_spatial)
        for method, use_ranks in (('cameraPR', False), ('cameraPR ranks', True)):
            t = time.perf_counter()
            TESTS[ds][method][label] = camera_pr(score, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=use_ranks, fdr=ALPHA)
            seconds[method] += time.perf_counter() - t
        for method, metric in (('GSEA', 'rank_normal'), ('GSEA raw', 'raw')):
            t = time.perf_counter()
            TESTS[ds][method][label] = rank_first_gsea(model.statistics.T_spatial, d['sets'], metric=metric,
                                                       min_size=SET_SIZES[0], max_size=SET_SIZES[1],
                                                       permutation_num=GSEA_PERMUTATIONS, seed=GSEA_SEED,
                                                       threads=min(N_JOBS, 32), fdr=ALPHA)
            seconds[method] += time.perf_counter() - t

    # Reference: matched rank-AUC (notebook 37), VIF from the two relabelings' matched z
    frames = {label: pd.DataFrame({'T_spatial': m.statistics.T_spatial.to_numpy()}, index=d['genes'])
              for label, m in MODELS[ds].items()}

    def matched_all(frames=frames, d=d, labels=labels):
        with parallel_config(backend='loky', inner_max_num_threads=1):
            parts = Parallel(n_jobs=len(labels))(delayed(_matched_one)(frames[p], d['sets'], d['strata']) for p in labels)
        table = pd.concat([t.assign(partition=p) for p, t in zip(labels, parts)], ignore_index=True)
        return {**{c: table[c].to_numpy().astype(str) for c in ('pathway_id', 'partition')},
                **{c: table[c].to_numpy(float) for c in ('n_genes', 'auc', 'effect', 'null_auc_mean', 'null_auc_sd')}}
    stored = cached(f'matched_{ds}', matched_all, params={'n_null': N_NULL, 'seed': SEED},
                    inputs={'scores': frames, 'sets': d['sets'], 'strata': d['strata']})
    matched = normal_matched_q(pd.DataFrame({k: (v.astype(str) if v.dtype.kind in 'OU' else v) for k, v in stored.items()}).assign(statistic='T_spatial'))
    t = time.perf_counter()
    null_z = pd.DataFrame({p: matched[matched.partition.eq(p)].set_index('pathway_id').z_matched.reindex(list(d['sets']))
                           for p in swaps})
    sizes = pd.Series({p: len(v) for p, v in d['sets'].items()})
    a, rho, vif = decoy_inflation(null_z, sizes)
    joint = joint_matched_test(matched.drop(columns=['z_matched']), vif)
    for label in labels:
        rows = joint[joint.partition.eq(label)].set_index('pathway_id').reindex(list(d['sets']))
        TESTS[ds]['joint'][label] = pd.DataFrame({'pathway_id': rows.index, 'effect': rows.effect.to_numpy(),
                                                  'z_joint': rows.z_joint.to_numpy(), 'p_one_sided': rows.p_joint.to_numpy(),
                                                  'q': rows.q_joint.to_numpy(),
                                                  'call': (rows.effect.gt(0) & rows.q_joint.le(ALPHA)).to_numpy()})
    cross = []
    for fit_on, judge in ((swaps[0], swaps[1]), (swaps[1], swaps[0])):
        _, _, vif_one = decoy_inflation(null_z[[fit_on]], sizes)
        cross.append(norm.sf(null_z[judge] / np.sqrt(vif_one.reindex(null_z.index))))
    seconds['joint'] = SECONDS[f'matched_{ds}'] + time.perf_counter() - t
    JOINT_FIT[ds] = {'a': float(a), 'rho': float(rho), 'median VIF': float(vif.median()),
                     'cross-fitted calibration share': float(np.mean(np.concatenate(cross) <= ALPHA))}
    RUNTIME[ds].update(seconds)

if 'human_vs_mouse' in JOINT_FIT:   # notebook 65 / pt-gene-model-nb.md: a = 1.04, rho = 0.0096 for this statistic
    fit = JOINT_FIT['human_vs_mouse']
    assert abs(fit['a'] - 1.04) < .01 and abs(fit['rho'] - .0096) < .0005, fit
    guard_log.append(f"human vs mouse: relabeling-fitted VIF a = {fit['a']:.3f}, ρ = {fit['rho']:.4f} = notebook 65's fit")
display(pd.DataFrame(JOINT_FIT).T.rename(index=lambda ds: DATA[ds]['title']).round(4))

A_CALLS = {ds: {m: {label: set(TESTS[ds][m][label].pathway_id[TESTS[ds][m][label].call.astype(bool)])
                    for label in PARTITIONS[ds]} for m in A_METHODS} for ds in DATASETS}
display(pd.DataFrame({(DATA[ds]['title'], SHORT[ds][label]): {A_METHODS[m]: len(A_CALLS[ds][m][label]) for m in A_METHODS}
                      for ds in DATASETS for label in PARTITIONS[ds]}))

## 4 · Method B: clustering-first (Caleb Hallinan's `pseudospace_reconstruction`, stages 06–07)

**What it does** (his package, his code, `compare.compare_groups` and `compare.pathways.run_pathways`):
1. **Gap curves.** Each group's curve is the mean of its specimens' log2 curves. The gap is
   log2(case / reference) along the grid, re-referenced to the typical gene (the median gap over
   genes is removed at every position). The same gap is computed for every case × reference
   specimen pair, and for every **mixed split** (one specimen of each group on each side), which is
   his null.
2. **Gene selection.** A gene is selected when its gap, over the whole axis or in its strongest
   local window, (i) reaches the 90th percentile of the mixed-split gaps of genes of similar
   expression, (ii) is at least twice its own largest mixed-split gap, and (iii) is reproduced at
   ≥ 50% by every specimen pair.
3. **Direction and shape.** Up, down or crossover; then uniform, gradient, peak, dip or local
   window, each shape passing the same three rules; and a location: a zone (S1, S2, S3) or a border.
4. **Clusters.** Ward clusters of the confirmed shapes within each direction. k is the largest
   ≤ 7 at which every cluster has ≥ 20 genes and every specimen pair recalls ≥ 50% of each cluster.
5. **ORA.** For every gene group (direction × shape × location) and every cluster, an
   expression-stratified hypergeometric test of each pathway, BH within library.

**One call means:** the pathway is over-represented (matched q < 0.05) in at least one gene group
or cluster: many of its genes were selected with one direction, shape and location.

**It also returns:** the gene groups and clusters themselves, with direction, shape and location;
and set curves with level-based "disrupted / coherent" calls (reported as a secondary line).

**What it cannot tell you:**
- a p-value per pathway: calls are conditional on the selected gene lists;
- anything about pathways whose genes change moderately but consistently, below the selection rules.

**How it is evaluated here.** His null for a run is its mixed splits. With 2 + 2 specimens there are
two, and they are exactly the other two balanced partitions. For the condition run these are the two
relabelings. For a relabeled run, one of them is the **condition split itself**, so the true effect
sits in the null and the run calls nothing by construction. Each relabeled run is therefore judged
with a **fair null** holding only the other relabeling. The condition run is reported with his
default null (both relabelings) and, like for like, with each relabeling alone.

In [ ]:
import pseudospace_reconstruction as prc
from pseudospace_reconstruction.compare import CompareParams, Design, compare_groups
from pseudospace_reconstruction.compare.pathways import PathwayParams, run_pathways
from pseudospace_reconstruction.features.axis import zone_cuts_from_markers

from pseudospace.pathway_pipelines import FLOOR_COUNTS, clustering_first_inputs, single_split_design, specimen_rate_floor

COMPARE_PARAMS = CompareParams(prior_k=0.)                                       # no prior in the log ratio
PATHWAY_PARAMS = PathwayParams(min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])    # his cap is 150
package = Path(prc.__file__).parent
external_key = digest([code_key, *sorted((package / 'compare').glob('*.py')), *sorted((package / 'features').glob('*.py')),
                       package / 'io' / 'gene_sets.py'])

ZONES, ZONE_ROWS, FLOOR_ROWS = {}, [], []
for ds, d in DATA.items():
    reference = np.flatnonzero(~d['case'])
    assert all(g in d['genes'] for genes in ZONE_MARKERS.values() for g in genes), 'a zone marker is not a tested gene'
    found = zone_cuts_from_markers(d['lognorm'][reference], d['genes'], d['position'][reference], ZONE_MARKERS)
    ZONES[ds] = np.asarray(found['cuts'], float)
    medians = pd.Series(d['position'][reference]).groupby(d['segment'][reference]).median()
    ZONE_ROWS.append({'dataset': d['title'], 'S1|S2 cut': ZONES[ds][0], 'S2|S3 cut': ZONES[ds][1],
                      'marker runs clean': found['runs_clean'], 'grid': f"[{MODELS[ds]['species'].grid[0]:.3f}, "
                      f"{MODELS[ds]['species'].grid[-1]:.3f}]",
                      **{f'median position of {s} structures': medians[s] for s in ('S1', 'S2', 'S3')}})
    floor = specimen_rate_floor(d['library'], d['specimen'])
    curves = MODELS[ds]['species'].specimen_curves
    below = np.any([curves[n] < floor[n] for n in curves], axis=0)
    shift = d['typical_shift']
    FLOOR_ROWS.append({'dataset': d['title'], 'genes where the half-count floor binds': int(below.any(axis=1).sum()),
                       'grid points floored (all genes)': int(below.sum()),
                       'typical-gene shift: min over grid (ln)': shift.min(), 'max (ln)': shift.max(),
                       'range (ln)': shift.max() - shift.min()})
ZONE_TABLE, FLOOR_TABLE = pd.DataFrame(ZONE_ROWS).set_index('dataset'), pd.DataFrame(FLOOR_ROWS).set_index('dataset')
ZONE_TABLE.to_csv(tables / 'method_b_zones.csv')
FLOOR_TABLE.to_csv(tables / 'method_b_floor_and_typical_gene_shift.csv')
display(ZONE_TABLE.round(3))
display(FLOOR_TABLE.round(3))

### 4.1 · What was standardized, and what was kept at his defaults

The table lists every change from his defaults. Everything not listed runs at his defaults:
selection rules, shape rules, Ward clustering and its k rule, ORA (10 expression strata, groups of
≥ 10 genes, ≥ 2 shared genes, BH within library, q < 0.05) and the set-curve rules.

In [ ]:
hm_floor, aki_floor = (FLOOR_TABLE.iloc[k]['genes where the half-count floor binds'] for k in range(2))
STANDARDIZATION = pd.DataFrame([
    ('Gene model', 'quasi-Poisson P-spline GAM on sample × position-bin pseudobulk; smoothing by GCV per gene; '
     'sample curves refitted at the pooled smoothing', 'the shared NB GLM on structures, fixed 6-df cubic B-spline, one α '
     'per gene; per-specimen curves (intercept + the same spline)', 'same curves and complexity as method A'),
    ('Rows of the fit', 'sample × position bins (0.001 wide in the reference run)', 'structures', 'no binning'),
    ('Exposure / offset', 'summed total counts of a bin', 'structure library: total over the measured orthologs', 'same offset as method A'),
    ('Grid', '201 points over the two groups\' common 1–99% support', f'{GRID_POINTS} points over the intersection of the four '
     'specimens\' 1st–99th percentile ranges', 'same grid; no specimen curve needs end-filling'),
    ('Gene universe', 'detected in ≥ 2% of the cohort\'s units, pooled', 'our universe (11,071 / 9,107 genes)', 'same genes as method A'),
    ('Gene sets', 'MSigDB Reactome/Hallmark v2026.1 + KEGG 2019, 10–150 genes', 'our 1,513 / 1,429 sets (Reactome 2022, Hallmark '
     '2020, KEGG 2019 Mouse), 10–300 genes (PathwayParams.max_genes = 300)', 'same sets as method A'),
    ('Prior in the log ratio', 'c = 0.3 × the gene\'s mean rate added to both rates (CompareParams.prior_k = 0.3)',
     f'none (prior_k = 0); each specimen curve floored at {FLOOR_COUNTS} count over the specimen\'s summed library',
     f'the floor binds for {hm_floor} (human vs mouse) and {aki_floor} (AKI) genes, all with almost no counts in a specimen'),
    ('Typical-gene reference', 'on: the median gap over genes is removed at every position', 'kept on: not exposed by his API',
     'shifts every gap by the typical gene; its size is in the table above'),
    ('Zones', 'marker rule (Slc5a12, Slc13a3, Slc16a9) on each run\'s own reference group', 'his marker rule, computed once on the '
     'control mice of the condition split and fixed for every run', 'one axis for all runs; relabelings have no biological reference group'),
    ('Border tolerance (node spacing)', 'from his stage-04 principal curve', '1/29: SCF13\'s 30-node curve, also used for the AKI DPT '
     'coordinate', 'only moves genes between a zone and its border label'),
    ('Expressed-elsewhere flags', 'computed from all label families', 'not computed', 'feed only his "without flagged genes" '
     'sensitivity table; no call uses them'),
    ('Grid weights', 'overlap of the two groups\' unit densities', 'kept: his rule, on our grid and structures', '—'),
    ('Null for a relabeled run', 'all mixed splits (contains the condition split)', 'his default reported; plus a single-split '
     '"fair" null (the other relabeling)', 'see section 4'),
], columns=['item', 'his default', 'here', 'effect / why'])
STANDARDIZATION.to_csv(tables / 'method_b_standardization.csv', index=False)
with pd.option_context('display.max_colwidth', 120):
    display(STANDARDIZATION)

### 4.2 · His runs

Seven runs per dataset, all on the same specimen curves:

| Contrast | Null (mixed splits) | Role |
|---|---|---|
| condition | both relabelings (his default) | his calls |
| condition | relabeling 1 only; relabeling 2 only | like-for-like with the relabeled runs |
| relabeling 1 | relabeling 2 only | fair relabeled run |
| relabeling 2 | relabeling 1 only | fair relabeled run |
| relabeling 1, relabeling 2 | his default (holds the condition split) | shows the zero by construction |

For each run the table gives the selected genes, the confirmed shapes, the clusters, his ORA calls
(by gene group, by cluster, either) and his set-curve calls.

In [ ]:
EXPRESSION_COLUMN = 'expression_per_10k'
GENE_COLUMNS = ['gene', 'selected', 'shape_replicated', 'direction', 'shape_class', 'location', 'group', 'cluster', 'rms',
                'level', 'shape_rms', EXPRESSION_COLUMN]


def design_of(ds, label):
    d = DATA[ds]
    per = pd.Series(PARTITIONS[ds][label][0], index=d['specimen']).groupby(level=0).first()
    return Design.from_labels(list(per.index), list(np.where(per.eq(1), 'case', 'reference')), case_label='case',
                              reference_label='reference')


def run_b(ds, contrast, null):
    """Caleb's stage 06 (compare_groups) and stage 07 (run_pathways) on the shared curves, as pipeline.run_stage06/07."""
    d = DATA[ds]
    axis, fit = clustering_first_inputs(MODELS[ds][contrast], d['position'], PARTITIONS[ds][contrast][0], d['specimen'],
                                        d['library'], zone_names=list(ZONE_MARKERS), zone_cuts=ZONES[ds],
                                        zone_short_names=ZONE_SHORT)
    design = design_of(ds, contrast)
    if null is not None:
        design = single_split_design(design, group_one(ds, null))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)     # his own all-NaN / inf guards inside shape and set curves
        result = compare_groups(fit, axis, design, node_spacing=NODE_SPACING, elsewhere_flags=None, params=COMPARE_PARAMS)
        gt = result.gene_table
        curves = result.curves()                            # float32, as his stage 07 reads them from curves.npz
        pathways = run_pathways(curves['D'].astype(float), curves['P'].astype(float), curves['N'].astype(float),
                                curves['weights'], curves['grid'], EXT_SETS[ds], gt[EXPRESSION_COLUMN].to_numpy(),
                                gt['group'].to_numpy(), cluster_labels=gt['cluster'].to_numpy(), flagged=None,
                                groups=list(result.shape_groups.index), pair_names=[str(n) for n in curves['pair_names']],
                                split_names=[str(n) for n in curves['null_names']], pair_samples=design.pairs,
                                params=PATHWAY_PARAMS)
    genes = gt[GENE_COLUMNS].copy()
    for c in ('selected', 'shape_replicated'):
        genes[c] = genes[c].fillna(False).astype(bool)
    for c in ('direction', 'shape_class', 'location', 'group', 'cluster'):
        genes[c] = genes[c].fillna('').astype(str)
    ora = pd.concat([t.assign(list=kind) for kind, t in (('group', pathways.by_group), ('cluster', pathways.by_cluster))],
                    ignore_index=True)
    ora = ora[['list', 'group', 'n_genes', 'library', 'pathway', 'set_size', 'overlap', 'fold_matched', 'p_plain', 'p_matched',
               'q_plain', 'q_matched', 'overlap_genes']].astype({'group': str, 'library': str, 'pathway': str, 'overlap_genes': str})
    ora.insert(0, 'pathway_id', ora.library + '::' + ora.pathway)
    calls = pathways.calls
    calls = pd.DataFrame({'pathway_id': (calls.library + '::' + calls.pathway).to_numpy(), 'pathway_call': calls.pathway_call.astype(str),
                          'is_representative': calls.is_representative.astype(bool), 'level_es': calls.level_es.astype(float),
                          'disruption_z_loo': calls.disruption_z_loo.astype(float), 'coherence': calls.coherence.astype(float)})
    info = pd.DataFrame({'null_names': [' | '.join(design.null_names)], 'n_splits': [len(design.mixed_splits)]})
    return {'genes': genes, 'ora': ora, 'calls': calls, 'info': info}


def cached_tables(stage, compute, *, params=None, inputs=None):
    """Cache a dict of DataFrames as one payload (columns as arrays; object columns as strings)."""
    def flat():
        out = {}
        for name, frame in compute().items():
            out[f'{name}|__columns'] = np.array(list(frame.columns), dtype=str)
            for column in frame.columns:
                values = frame[column].to_numpy()
                out[f'{name}|{column}'] = values.astype(str) if values.dtype == object else values
        return out
    payload = cached(stage, flat, params=params, inputs=inputs, key=external_key)
    names = sorted({k.split('|', 1)[0] for k in payload})
    return {n: pd.DataFrame({c: payload[f'{n}|{c}'] for c in payload[f'{n}|__columns'].astype(str)}) for n in names}


B, B_STAGE = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    condition, r1, r2 = labels
    runs = [(condition, None), (condition, r1), (condition, r2), (r1, r2), (r2, r1), (r1, None), (r2, None)]
    curves_key = digest({n: c for n, c in MODELS[ds]['species'].specimen_curves.items()})
    for contrast, null in runs:
        stage = f'method_b_{ds}_{slug(contrast)}__null_{"default" if null is None else slug(null)}'
        B[ds][(contrast, null)] = cached_tables(stage, lambda ds=ds, contrast=contrast, null=null: run_b(ds, contrast, null),
                                                params={'compare': repr(COMPARE_PARAMS), 'pathways': repr(PATHWAY_PARAMS),
                                                        'node_spacing': NODE_SPACING, 'floor': FLOOR_COUNTS},
                                                inputs={**d['input_key'], 'curves': curves_key, 'group': PARTITIONS[ds][contrast][0],
                                                        'zones': ZONES[ds], 'sets': d['sets'],
                                                        'null': 'default' if null is None else group_one(ds, null)})
        B_STAGE[ds][(contrast, null)] = stage


def b_lists(t):
    """His pathway outputs of one run: ORA calls by group, by cluster, either; set-curve calls."""
    sig = t['ora'][t['ora'].q_matched.astype(float) < ALPHA]
    group, cluster = set(sig.pathway_id[sig.list.eq('group')]), set(sig.pathway_id[sig.list.eq('cluster')])
    call = t['calls'].pathway_call
    return {'ORA group': group, 'ORA cluster': cluster, 'ORA': group | cluster,
            'set curve': set(t['calls'].pathway_id[~call.isin(['not disrupted', 'not evaluated'])]),
            'set curve coherent': set(t['calls'].pathway_id[call.str.startswith('coherent')])}


def b_bonferroni_p(ds, t):
    """Notebook 61's convention for list-based methods: the best matched ORA p over his lists x the number of lists."""
    genes = t['genes']
    n_lists = sum(int((genes[c][genes[c].ne('')].value_counts() >= PATHWAY_PARAMS.min_group_size).sum()) for c in ('group', 'cluster'))
    best = t['ora'].groupby('pathway_id').p_matched.min().astype(float)
    return (best * max(n_lists, 1)).clip(upper=1.).reindex(list(DATA[ds]['sets'])).fillna(1.).to_numpy()


def null_name(ds, null):
    return 'default (both other partitions)' if null is None else SHORT[ds][null]


RUN_ROWS = []
for ds in DATASETS:
    for (contrast, null), t in B[ds].items():
        genes, lists = t['genes'], b_lists(t)
        selected = genes[genes.selected]
        clusters = selected.cluster[selected.cluster.ne('') & ~selected.cluster.str.endswith('0')]
        RUN_ROWS.append({'dataset': DATA[ds]['title'], 'contrast': SHORT[ds][contrast], 'null': null_name(ds, null),
                         'selected genes': len(selected), 'shape confirmed': int(genes.shape_replicated.sum()),
                         'up / down / crossover': '{} / {} / {}'.format(*(int(selected.direction.eq(x).sum()) for x in ('up', 'down', 'crossover'))),
                         'clusters (excluding *0)': clusters.nunique(), 'gene groups ≥ 10 genes': int((selected.group.value_counts() >= 10).sum()),
                         'ORA calls: group': len(lists['ORA group']), 'cluster': len(lists['ORA cluster']), 'either': len(lists['ORA']),
                         'either, representatives': len(representatives(ds, lists['ORA'])),
                         'set-curve calls': len(lists['set curve']), 'of which coherent': len(lists['set curve coherent']),
                         'seconds': round(SECONDS[B_STAGE[ds][(contrast, null)]], 1)})
RUN_TABLE = pd.DataFrame(RUN_ROWS)
RUN_TABLE.to_csv(tables / 'method_b_runs.csv', index=False)
display(RUN_TABLE)

## 5 · Comparison

### 5.1 · The method tables

For each method and dataset:
- **calls:** pathways called for the condition split; in brackets, the redundancy clusters they touch;
- **relabeled calls:** calls under the two relabelings (method B: with the fair, single-split null);
- **NCDR:** mean relabeled calls ÷ condition calls;
- **relabeling rule** (notebook 31): NCDR < 0.25 and each relabeling below 5% of tested pathways;
- **calibration share:** of the 2 × (tested pathways) pooled relabeled one-sided p-values, the share
  ≤ 0.05; a calibrated test gives about 0.05, and the limit is 0.075 (notebook 61's criterion 2).
  Method B has no p-value per pathway. Its line uses notebook 61's convention for list-based methods,
  the best matched ORA p over his lists times the number of lists. It is nearly always 1 under a
  relabeling, because almost no gene is selected there, so it passes trivially;
- **seconds:** compute time of the pathway step for the three partitions, after the shared gene
  model (whose time is printed separately).

**What it can tell you:** how many pathways each method calls and how specific the calls are against
specimen noise. **What it cannot tell you:** which calls are right. A relabeling mixes the two
conditions, so it cannot see an artefact that follows the condition (depth, probe panels, genes
absent in one group). Section 5.4 looks at that.

In [ ]:
RETURNS = {'cameraPR': 'per pathway: direction, p, FDR', 'cameraPR ranks': 'per pathway: direction, p, FDR',
           'GSEA': 'per pathway: NES, p, FDR, leading-edge genes', 'GSEA raw': 'per pathway: NES, p, FDR, leading-edge genes',
           'joint': 'per pathway: matched AUC, z, p, q',
           'ORA': 'gene groups (direction × shape × location) and clusters with their enriched pathways; no p per pathway',
           'ORA single': 'as above', 'set curve': 'per pathway: level-based disruption, coherence and direction'}
METHOD_ROWS, PLOT_ROWS = [], []
for ds in DATASETS:
    d = DATA[ds]
    condition, r1, r2 = list(PARTITIONS[ds])
    tested = len(d['sets'])

    def add(key, label, family, condition_sets, relabeled_sets, p_relabeled, seconds, note=''):
        n_condition = [len(c) for c in condition_sets]
        n_relabeled = [len(r) for r in relabeled_sets]
        mean_condition = float(np.mean(n_condition))
        ncdr = np.mean(n_relabeled) / mean_condition if mean_condition else np.nan
        share = float(np.mean(np.asarray(p_relabeled, float) <= ALPHA)) if p_relabeled is not None else np.nan
        METHOD_ROWS.append({
            'dataset': d['title'], 'method': label, 'family': family,
            'calls': ', '.join(map(str, n_condition)),
            'calls (redundancy clusters)': ', '.join(str(len(representatives(ds, c))) for c in condition_sets),
            'relabeled calls': ', '.join(map(str, n_relabeled)), 'NCDR': ncdr,
            'relabeling rule': 'pass' if mean_condition and ncdr < NCDR_MAX and max(n_relabeled) < NULL_RATE * tested else 'fail',
            'calibration share': share,
            'calibration': ('n/a' if p_relabeled is None else 'pass' if share <= CALIBRATION_MAX else 'fail') + note,
            'seconds': round(seconds, 1), 'returns': RETURNS[key]})
        PLOT_ROWS.append({'dataset': ds, 'method': label, 'family': family, 'condition': n_condition,
                          'relabeled': n_relabeled, 'tested': tested})

    for m, label in A_METHODS.items():
        add(m, label, 'reference' if m == 'joint' else 'A', [A_CALLS[ds][m][condition]],
            [A_CALLS[ds][m][r1], A_CALLS[ds][m][r2]],
            np.concatenate([TESTS[ds][m][r].p_one_sided.to_numpy(float) for r in (r1, r2)]), RUNTIME[ds][m],
            ' (VIF fitted on these relabelings)' if m == 'joint' else '')
    fair = [B[ds][(r1, r2)], B[ds][(r2, r1)]]
    fair_seconds = sum(SECONDS[B_STAGE[ds][k]] for k in ((r1, r2), (r2, r1)))
    add('ORA', 'B · ORA on gene groups and clusters (his default null)', 'B', [b_lists(B[ds][(condition, None)])['ORA']],
        [b_lists(t)['ORA'] for t in fair], np.concatenate([b_bonferroni_p(ds, t) for t in fair]),
        SECONDS[B_STAGE[ds][(condition, None)]] + fair_seconds, ' (list-based convention)')
    add('ORA single', 'B · ORA, single-split null (like for like)', 'B',
        [b_lists(B[ds][(condition, r)])['ORA'] for r in (r1, r2)], [b_lists(t)['ORA'] for t in fair],
        np.concatenate([b_bonferroni_p(ds, t) for t in fair]),
        sum(SECONDS[B_STAGE[ds][(condition, r)]] for r in (r1, r2)) + fair_seconds, ' (list-based convention)')
    add('set curve', 'B · set-curve calls, level-based (secondary)', 'B', [b_lists(B[ds][(condition, None)])['set curve']],
        [b_lists(t)['set curve'] for t in fair], None, SECONDS[B_STAGE[ds][(condition, None)]] + fair_seconds)
METHOD_TABLE = pd.DataFrame(METHOD_ROWS)
METHOD_TABLE.to_csv(tables / 'method_table.csv', index=False)
GENE_MODEL_SECONDS = {ds: sum(SECONDS[f'gene_model_{ds}_{slug(p)}'] for p in PARTITIONS[ds]) for ds in DATASETS}
for ds in DATASETS:
    print(f"{DATA[ds]['title']}: {len(DATA[ds]['sets']):,} pathways tested; shared gene model {GENE_MODEL_SECONDS[ds]:.0f} s "
          f"(three partitions, {N_JOBS} workers)")
    with pd.option_context('display.max_colwidth', 60):
        display(METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].drop(columns='dataset').set_index('method').round(3))
for ds in DATASETS:
    fit = JOINT_FIT[ds]
    print(f"{DATA[ds]['title']}: reference joint test, relabeling-fitted VIF a = {fit['a']:.3f}, ρ = {fit['rho']:.4f}, "
          f"median VIF {fit['median VIF']:.2f}; cross-fitted calibration share {fit['cross-fitted calibration share']:.3f}")

**The figure.** One row per method. Filled dots: the condition split's calls (two dots for the
single-split row, one per null). Open dots: the two relabelings. The dashed line marks 5% of the
tested pathways, the most one relabeling may call. A specific method has its filled dot far to the
right of its open dots. Right margin: NCDR and the calibration share.

In [ ]:
FAMILY_COLOR = {'A': A_COLOR, 'B': B_COLOR, 'reference': REF_COLOR}
fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 66 * MM), sharey=True, gridspec_kw={'wspace': .62})
order = list(dict.fromkeys(r['method'] for r in PLOT_ROWS))
for ax, ds in zip(axes, DATASETS):
    rows = {r['method']: r for r in PLOT_ROWS if r['dataset'] == ds}
    table = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    for i, method in enumerate(order):
        y, r = len(order) - 1 - i, rows[method]
        color = FAMILY_COLOR[r['family']]
        ax.plot([0, max(r['condition'] + r['relabeled'])], [y, y], color=LIGHT, lw=.6, zorder=1)
        ax.scatter(r['relabeled'], [y] * 2, s=22, facecolor='white', edgecolor=color, linewidth=1, zorder=3)
        ax.scatter(r['condition'], [y] * len(r['condition']), s=22, color=color, edgecolor='white', linewidth=.5, zorder=4)
        row = table.loc[method]
        cal = '—' if np.isnan(row['calibration share']) else f"{row['calibration share']:.3f}"
        flag = [] if row['relabeling rule'] == 'pass' else ['fails relabeling']
        flag += ['fails calibration'] if row['calibration'].startswith('fail') else []
        ncdr = '—' if np.isnan(row['NCDR']) else f"{row['NCDR']:.2f}"
        ax.text(1.02, y, f"NCDR {ncdr} · cal. {cal}" + (f"\n{', '.join(flag)}" if flag else ''),
                transform=ax.get_yaxis_transform(), va='center', fontsize=5.2, color=INK, linespacing=1.1)
    ax.axvline(NULL_RATE * rows[order[0]]['tested'], color=MUTED, lw=.6, ls='--', zorder=0)
    ax.text(NULL_RATE * rows[order[0]]['tested'], -.75, ' 5% of tested', fontsize=5, color=MUTED, va='center')
    ax.set_xlim(left=-5)
    ax.set_ylim(-1.1, len(order) - .5)
    ax.set_xlabel('pathways called')
    ax.set_title(DATA[ds]['title'], loc='left')
axes[0].set_yticks(range(len(order)), order[::-1])
for tick, method in zip(axes[0].get_yticklabels(), order[::-1]):
    tick.set_color(FAMILY_COLOR[next(r['family'] for r in PLOT_ROWS if r['method'] == method)])
handles = [plt.Line2D([], [], marker='o', lw=0, color=INK, markersize=4, label='condition split'),
           plt.Line2D([], [], marker='o', lw=0, markerfacecolor='white', markeredgecolor=INK, markersize=4,
                      label='relabelings (method B: fair single-split null)')]
fig.legend(handles=handles, loc='upper center', ncol=2, bbox_to_anchor=(.5, -.04))
save(fig, 'fig1_method_tables')
plt.show()

### 5.2 · Why method A's set tests call pathways under relabeling (post hoc)

This subsection was added after the method tables showed that cameraPR and GSEA fail the relabeling
rule. It asks why, and it changes no call above.

**Two candidate reasons.**
1. **Power follows expression.** The NB `T_spatial` of a well-measured gene is larger in any
   partition, relabelings included, because more counts give more power. A pathway of highly
   expressed genes then ranks high whatever the grouping. cameraPR and GSEA compare a pathway with
   all other genes; our joint test compares it with genes matched on expression, detection and
   coverage.
2. **Correlation between a pathway's genes.** cameraPR assumes `inter.gene.cor = 0.01`. The table
   gives the inflation its own relabeled z-scores imply (the fit E[z²] = a(1 + (m − 1)ρ), as for the
   joint test).

**The check.** cameraPR is rerun on scores rank-normalised **within** the joint test's 27 matching
strata (expression × detection × coverage tertiles), so every stratum contributes the same score
distribution. If expression-linked power is the reason, this version should call far fewer
pathways under relabeling. It is a diagnostic, not a new primary.

In [ ]:
from scipy.stats import rankdata, spearmanr


def within_stratum_scores(statistic, strata):
    # Rank-normal scores computed separately inside each matching stratum.
    values = pd.Series(np.asarray(statistic, float), index=statistic.index)
    return values.groupby(np.asarray(strata)).transform(lambda v: pd.Series(norm.ppf((rankdata(v) - .5) / len(v)), index=v.index))


WHY_ROWS, POSTHOC = [], {ds: {} for ds in DATASETS}
for ds, d in DATA.items():
    condition, r1, r2 = list(PARTITIONS[ds])
    sizes = pd.Series({p: len(v) for p, v in d['sets'].items()})
    gene_rows = {}
    for label, model in MODELS[ds].items():
        score = rank_normal_scores(model.statistics.T_spatial)
        between = score.groupby(np.asarray(d['strata'])).transform('mean')
        gene_rows[label] = (spearmanr(model.statistics.T_spatial, d['mean_expression'])[0],
                            float(between.var() / score.var()))
        POSTHOC[ds][label] = camera_pr(within_stratum_scores(model.statistics.T_spatial, d['strata']), d['sets'],
                                       inter_gene_cor=CAMERA_COR, fdr=ALPHA)
    for key, tests in (('cameraPR (primary)', TESTS[ds]['cameraPR']), ('cameraPR, within-stratum scores (post hoc)', POSTHOC[ds])):
        z = pd.DataFrame({r: tests[r].set_index('pathway_id').p_one_sided.clip(1e-300, 1 - 1e-16).map(norm.isf) for r in (r1, r2)})
        a, rho, _ = decoy_inflation(z, sizes)
        calls = [int(tests[p].call.sum()) for p in (condition, r1, r2)]
        share = float(np.mean(np.concatenate([tests[r].p_one_sided.to_numpy(float) for r in (r1, r2)]) <= ALPHA))
        WHY_ROWS.append({'dataset': d['title'], 'test': key, 'calls': calls[0], 'relabeled calls': f'{calls[1]}, {calls[2]}',
                         'NCDR': np.mean(calls[1:]) / calls[0] if calls[0] else np.nan, 'calibration share': share,
                         'implied a': a, 'implied ρ': rho,
                         **{f'Spearman T_spatial vs expression · {SHORT[ds][p]}': gene_rows[p][0] for p in (condition, r1, r2)},
                         **{f'score variance between strata · {SHORT[ds][p]}': gene_rows[p][1] for p in (condition, r1, r2)}})
WHY = pd.DataFrame(WHY_ROWS)
WHY.to_csv(tables / 'why_method_a_is_not_specific.csv', index=False)
display(WHY.set_index(['dataset', 'test']).iloc[:, :6].round(3))
display(WHY.drop_duplicates('dataset').set_index('dataset').iloc[:, 7:].round(3))

### 5.3 · Overlap between the methods

Each pair of call sets (condition split) is compared twice: over the called pathways, and over the
redundancy clusters they touch (the rule of section 1.2, fixed before any result). The main
comparison is method A's primary, cameraPR, against method B's pathway output, his ORA calls.

In [ ]:
def call_set(ds, key):
    condition = list(PARTITIONS[ds])[0]
    if key == 'B ORA':
        return b_lists(B[ds][(condition, None)])['ORA']
    if key == 'B set curve':
        return b_lists(B[ds][(condition, None)])['set curve']
    return A_CALLS[ds][key][condition]


LABEL = {'cameraPR': 'A cameraPR', 'GSEA': 'A GSEA', 'joint': 'reference joint', 'B ORA': 'B ORA', 'B set curve': 'B set curve'}
PAIRS = [('cameraPR', 'B ORA'), ('GSEA', 'B ORA'), ('joint', 'B ORA'), ('cameraPR', 'B set curve'), ('cameraPR', 'joint'),
         ('cameraPR', 'GSEA')]
OVERLAP_ROWS = []
for ds in DATASETS:
    for left, right in PAIRS:
        for unit, collapse in (('pathways', lambda s, ds=ds: s), ('redundancy clusters', lambda s, ds=ds: representatives(ds, s))):
            x, y = collapse(call_set(ds, left)), collapse(call_set(ds, right))
            OVERLAP_ROWS.append({'dataset': DATA[ds]['title'], 'comparison': f'{LABEL[left]} vs {LABEL[right]}', 'unit': unit,
                                 'left': len(x), 'right': len(y), 'left only': len(x - y), 'both': len(x & y),
                                 'right only': len(y - x), 'Jaccard': len(x & y) / len(x | y) if x | y else np.nan})
OVERLAP = pd.DataFrame(OVERLAP_ROWS)
OVERLAP.to_csv(tables / 'overlap.csv', index=False)
display(OVERLAP.pivot_table(index=['dataset', 'comparison'], columns='unit',
                            values=['left only', 'both', 'right only', 'Jaccard'], sort=False).round(3))

fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 52 * MM), sharey=True, gridspec_kw={'wspace': .08})
for ax, ds in zip(axes, DATASETS):
    rows = OVERLAP[OVERLAP.dataset.eq(DATA[ds]['title']) & OVERLAP.unit.eq('redundancy clusters')].reset_index(drop=True)
    for i, r in rows.iterrows():
        y = len(rows) - 1 - i
        left, right = PAIRS[i]
        colors = [FAMILY_COLOR['reference' if left == 'joint' else 'A'], INK,
                  FAMILY_COLOR['reference' if right == 'joint' else 'B' if right.startswith('B') else 'A']]
        start = 0
        for value, color in zip((r['left only'], r['both'], r['right only']), colors):
            ax.barh(y, value, left=start, color=color, height=.6, edgecolor='white', linewidth=.8)
            if value >= .045 * rows[['left only', 'both', 'right only']].sum(axis=1).max():
                ax.text(start + value / 2, y, str(int(value)), ha='center', va='center', fontsize=5,
                        color='white' if color in (INK, A_COLOR, REF_COLOR) else INK)
            start += value
        ax.text(start + 3, y, f"J = {r['Jaccard']:.2f}", va='center', fontsize=5.2, color=MUTED)
    ax.set_yticks(range(len(rows)), [f'{LABEL[a]} | both | {LABEL[b]}' for a, b in PAIRS][::-1])
    ax.set_xlabel('redundancy clusters called (condition split)')
    ax.set_title(DATA[ds]['title'], loc='left')
    ax.margins(x=.12)
save(fig, 'fig2_overlap')
plt.show()

### 5.4 · What each method finds that the other does not

For cameraPR (method A) against his ORA (method B), the pathways only one of them calls, collapsed to
redundancy clusters. Each cluster is listed by its called pathway with the smallest q of the method
that calls it. For each pathway:
- **share nearly absent:** the share of its genes detected in < 1% of one group's structures;
- **median |log ratio|:** the median over its genes of the mean NB log rate ratio, case / reference,
  over the grid (natural log): the size of the offset;
- **mean score:** the mean rank-normal `T_spatial` score of its genes (0 is a typical gene);
- **B selected:** the share of its genes his selection keeps, how many of his gene groups they
  fall into, and his best list (gene group or cluster) with its matched q.

His lists are of two kinds. A list **with a confirmed shape** is a gradient, peak, dip or crossover
group, or a numbered shape cluster. A list **without** one is an "uniform" group, a "local window"
group or a cluster `U0`/`D0`/`X0` (selected genes whose shape did not replicate). The last table
counts his calls by the kind of list that carries them, and the next one lists the five lists
that carry most of his calls.

The composition table summarizes every call set with the same descriptors (medians over its
pathways), and the gene-level lines compare the genes each method starts from: method A's top 10%
by `T_spatial`, and method B's selected genes.

In [ ]:
DETAIL, LEADS, COMPOSITION_ROWS, GENE_ROWS, LIST_KIND_ROWS, LIST_TOP_ROWS = {}, {}, [], [], [], []
for ds, d in DATA.items():
    condition = list(PARTITIONS[ds])[0]
    gene_index = pd.Index(d['genes'])
    members = {p: gene_index.get_indexer(v) for p, v in d['sets'].items()}
    score = rank_normal_scores(MODELS[ds]['species'].statistics.T_spatial).to_numpy()
    abs_ratio = d['log_ratio'].abs().to_numpy()
    b_condition = B[ds][(condition, None)]
    selected = b_condition['genes'].set_index('gene').selected.reindex(d['genes']).to_numpy(bool)
    ora = b_condition['ora'].astype({'q_matched': float, 'overlap': int})
    best = ora.sort_values(['q_matched', 'overlap'], ascending=[True, False]).drop_duplicates('pathway_id').set_index('pathway_id')
    camera = TESTS[ds]['cameraPR'][condition].set_index('pathway_id')
    detail = pd.DataFrame(index=pd.Index(list(d['sets']), name='pathway_id'))
    detail['name'] = NAMES[ds].reindex(detail.index)
    detail['n genes'] = [len(members[p]) for p in detail.index]
    detail['redundancy cluster'] = REDUNDANCY[ds].cluster_id.reindex(detail.index)
    detail['cameraPR direction'] = camera.direction.reindex(detail.index)
    detail['cameraPR FDR'] = camera.fdr.reindex(detail.index)
    detail['GSEA q'] = TESTS[ds]['GSEA'][condition].set_index('pathway_id').q.reindex(detail.index)
    detail['joint q'] = TESTS[ds]['joint'][condition].set_index('pathway_id').q.reindex(detail.index)
    detail['B best list'] = (best.list + ': ' + best.group).reindex(detail.index)
    detail['B best q'] = best.q_matched.reindex(detail.index)
    detail['B overlap'] = best.overlap.reindex(detail.index)
    detail['B share selected'] = [float(selected[members[p]].mean()) for p in detail.index]
    gene_group = b_condition['genes'].set_index('gene').group.reindex(d['genes']).to_numpy()
    detail['B groups holding its genes'] = [len(set(gene_group[members[p]]) - {''}) for p in detail.index]
    sig = ora[ora.q_matched < ALPHA].copy()
    sig['shape list'] = np.where(sig.list.eq('cluster'), ~sig.group.str.endswith('0'),
                                 ~sig.group.str.contains('uniform|local window'))
    via_shape = sig.groupby('pathway_id')['shape list'].any()
    detail['B via a confirmed-shape list'] = via_shape.reindex(detail.index)
    detail['share nearly absent'] = [float(d['nearly_absent'][members[p]].mean()) for p in detail.index]
    detail['median |log ratio|'] = [float(np.median(abs_ratio[members[p]])) for p in detail.index]
    detail['mean score'] = [float(score[members[p]].mean()) for p in detail.index]
    for key in ('cameraPR', 'GSEA', 'joint', 'B ORA', 'B set curve'):
        detail[f'called: {LABEL[key]}'] = detail.index.isin(sorted(call_set(ds, key)))
    detail.to_csv(tables / f'pathway_detail_{ds}.csv')
    DETAIL[ds] = detail

    a_calls, b_calls = call_set(ds, 'cameraPR'), call_set(ds, 'B ORA')
    a_only_clusters = set(detail.loc[sorted(a_calls), 'redundancy cluster']) - set(detail.loc[sorted(b_calls), 'redundancy cluster'])
    b_only_clusters = set(detail.loc[sorted(b_calls), 'redundancy cluster']) - set(detail.loc[sorted(a_calls), 'redundancy cluster'])
    lead_a = (detail.loc[sorted(a_calls)].query('`redundancy cluster` in @a_only_clusters')
              .sort_values(['cameraPR FDR', 'name']).drop_duplicates('redundancy cluster'))
    lead_b = (detail.loc[sorted(b_calls)].query('`redundancy cluster` in @b_only_clusters')
              .sort_values(['B best q', 'B overlap', 'name'], ascending=[True, False, True]).drop_duplicates('redundancy cluster'))
    LEADS[ds] = {'A only': lead_a, 'B only': lead_b}
    columns = ['name', 'n genes', 'cameraPR FDR', 'B best list', 'B best q', 'B share selected', 'B groups holding its genes',
               'share nearly absent', 'median |log ratio|', 'mean score']
    for side, lead in LEADS[ds].items():
        lead[columns].to_csv(tables / f'only_{"cameraPR" if side == "A only" else "B_ORA"}_{ds}.csv')
        print(f"\n{DATA[ds]['title']} · {'cameraPR only' if side == 'A only' else 'his ORA only'}: "
              f"{len(lead)} redundancy clusters ({'smallest cameraPR FDR' if side == 'A only' else 'smallest B q'} first)")
        shown = lead[columns].head(12).round(3)
        shown[['cameraPR FDR', 'B best q']] = lead[['cameraPR FDR', 'B best q']].head(12).map(lambda v: f'{v:.1e}')
        with pd.option_context('display.max_colwidth', 50):
            display(shown.set_index('name'))

    sets_ = {'A cameraPR': a_calls, 'B ORA': b_calls, 'cameraPR only': a_calls - b_calls, 'both': a_calls & b_calls,
             'B ORA only': b_calls - a_calls, 'A GSEA': call_set(ds, 'GSEA'), 'reference joint': call_set(ds, 'joint'),
             'B set curve': call_set(ds, 'B set curve'), 'all pathways': set(d['sets'])}
    for name, s in sets_.items():
        rows = detail.loc[sorted(s)]
        COMPOSITION_ROWS.append({'dataset': d['title'], 'call set': name, 'pathways': len(rows),
                                 'median share nearly absent': rows['share nearly absent'].median(),
                                 'median |log ratio|': rows['median |log ratio|'].median(),
                                 'median mean score': rows['mean score'].median()})
    per_list = sig.groupby(['list', 'group']).agg(genes=('n_genes', 'first'), pathways=('pathway_id', 'nunique'))
    for (kind, name), row in per_list.sort_values('pathways', ascending=False).head(5).iterrows():
        LIST_TOP_ROWS.append({'dataset': d['title'], 'his list': f'{kind}: {name}', 'genes in the list': int(row.genes),
                              'pathways called through it': int(row.pathways),
                              'share of his calls': row.pathways / len(b_calls),
                              'redundancy clusters': len(representatives(ds, set(sig.pathway_id[sig.list.eq(kind) & sig.group.eq(name)])))})
    b_calls_detail = detail.loc[sorted(b_calls)]
    for part, rows in (('all his ORA calls', b_calls_detail), ('his ORA only', b_calls_detail.loc[sorted(b_calls - a_calls)]),
                       ('also called by cameraPR', b_calls_detail.loc[sorted(b_calls & a_calls)])):
        LIST_KIND_ROWS.append({'dataset': d['title'], 'his ORA calls': part, 'pathways': len(rows),
                               'via a confirmed-shape list': int(rows['B via a confirmed-shape list'].astype(bool).sum()),
                               'only via lists without a confirmed shape': int((~rows['B via a confirmed-shape list'].astype(bool)).sum())})
    top = score >= np.quantile(score, .9)
    for name, mask in (('all genes', np.ones(len(score), bool)), ('A: top 10% by T_spatial', top),
                       ('B: selected genes', selected), ('B: selected, shape confirmed',
                                                         b_condition['genes'].set_index('gene').shape_replicated.reindex(d['genes']).to_numpy(bool))):
        GENE_ROWS.append({'dataset': d['title'], 'genes': name, 'n': int(mask.sum()),
                          'share nearly absent': float(d['nearly_absent'][mask].mean()),
                          'median |log ratio|': float(np.median(abs_ratio[mask])),
                          'median score': float(np.median(score[mask]))})
COMPOSITION = pd.DataFrame(COMPOSITION_ROWS)
GENE_COMPOSITION = pd.DataFrame(GENE_ROWS)
LIST_KIND, LIST_TOP = pd.DataFrame(LIST_KIND_ROWS), pd.DataFrame(LIST_TOP_ROWS)
LIST_KIND.to_csv(tables / 'method_b_calls_by_list_kind.csv', index=False)
LIST_TOP.to_csv(tables / 'method_b_top_lists.csv', index=False)
COMPOSITION.to_csv(tables / 'call_set_composition.csv', index=False)
GENE_COMPOSITION.to_csv(tables / 'gene_list_composition.csv', index=False)
display(COMPOSITION.set_index(['dataset', 'call set']).round(3))
display(GENE_COMPOSITION.set_index(['dataset', 'genes']).round(3))
display(LIST_KIND.set_index(['dataset', 'his ORA calls']))
display(LIST_TOP.set_index(['dataset', 'his list']).round(3))

### 5.5 · Worked examples

**Choice rule (fixed before looking).** For each dataset, the two pathways at the top of each
"only" table of section 5.4: cameraPR-only by the smallest cameraPR FDR, his-ORA-only by the
smallest matched q (ties: the larger overlap).

**Each panel.**
- **Heatmap:** the **shape** of every member gene's difference curve: Δ(s) minus its mean over the
  grid, where Δ(s) is the NB log rate ratio case / reference (natural log, clipped at ±1). Rows are
  ordered by where the shape peaks. Colour changing along a row is what `T_spatial` measures.
- **Right strips:**
  - **offset:** the gene's mean Δ, the part the heatmap removes (clipped at ±2);
  - **score:** its rank-normal `T_spatial` score, method A's input (darker = higher);
  - **B:** whether his selection keeps the gene, method B's input (orange = selected).
- Thin lines mark his S1|S2 and S2|S3 zone cuts.
- **Title:** the pathway, and one line on why one method calls it and the other does not.

In [ ]:
import textwrap

from matplotlib.colors import LinearSegmentedColormap, Normalize

DIVERGING = LinearSegmentedColormap.from_list('delta', ['#2a78d6', '#f0efec', '#e34948'])
SCORE_MAP = LinearSegmentedColormap.from_list('score', ['#f0efec', A_COLOR])
SELECT_MAP = LinearSegmentedColormap.from_list('selected', ['#f0efec', B_COLOR])
EXAMPLES = []
for ds in DATASETS:
    for side in ('A only', 'B only'):
        for pathway in LEADS[ds][side].index[:2]:
            EXAMPLES.append((ds, side, pathway))


def reason(ds, side, pathway):
    r = DETAIL[ds].loc[pathway]
    if side == 'A only':
        q = 'no list reaches q < 0.05' if not r['B best q'] < ALPHA else f"best list q = {r['B best q']:.2g}"
        return (f"cameraPR FDR {r['cameraPR FDR']:.1e}; genes score {r['mean score']:+.2f}. B keeps {r['B share selected']:.0%} "
                f"of them but spreads them over {int(r['B groups holding its genes'])} gene groups; {q}")
    return (f"B {r['B best list']} (q = {r['B best q']:.1e}, {int(r['B overlap'])} genes). Genes score {r['mean score']:+.2f}, "
            f"median |offset| {r['median |log ratio|']:.2f}; cameraPR FDR {r['cameraPR FDR']:.2f}")


fig = plt.figure(figsize=(180 * MM, 168 * MM))
outer = fig.add_gridspec(2, 4, hspace=.85, wspace=.38, left=.04, right=.99, top=.94, bottom=.15)
for k, (ds, side, pathway) in enumerate(EXAMPLES):
    d, model = DATA[ds], MODELS[ds]['species']
    idx = pd.Index(d['genes']).get_indexer(d['sets'][pathway])
    shape = model.delta[idx] - model.delta[idx].mean(axis=1, keepdims=True)
    order = np.lexsort((-shape.max(axis=1), np.argmax(shape, axis=1)))
    idx, shape = idx[order], shape[order]
    score = rank_normal_scores(model.statistics.T_spatial).to_numpy()
    selected = B[ds][(list(PARTITIONS[ds])[0], None)]['genes'].set_index('gene').selected.reindex(d['genes']).to_numpy(bool)
    cell = outer[k // 4, k % 4].subgridspec(1, 4, width_ratios=[12, 1, 1, 1], wspace=.12)
    ax = fig.add_subplot(cell[0])
    grid = model.grid
    shape_image = ax.imshow(np.clip(shape, -1, 1), aspect='auto', cmap=DIVERGING, norm=Normalize(-1, 1),
                            extent=(grid[0], grid[-1], len(idx), 0), interpolation='nearest')
    for cut in ZONES[ds]:
        ax.axvline(cut, color=INK, lw=.4, alpha=.6)
    for name, lo, hi in zip(('S1', 'S2', 'S3'), [grid[0], *ZONES[ds]], [*ZONES[ds], grid[-1]]):
        ax.text((lo + hi) / 2, 0, name, ha='center', va='bottom', fontsize=5, color=MUTED, clip_on=False)
    ax.set_yticks([])
    ax.set_xticks([grid[0], grid[-1]], [f'{grid[0]:.2f}', f'{grid[-1]:.2f}'])
    for label, align in zip(ax.get_xticklabels(), ('left', 'right')):
        label.set_horizontalalignment(align)
    ax.set_xlabel('position along the PT', labelpad=1)
    ax.set_ylabel(f'{len(idx)} genes', labelpad=1)
    tag = 'cameraPR only' if side == 'A only' else 'his ORA only'
    name = '\n'.join(textwrap.wrap(textwrap.shorten(NAMES[ds][pathway], 60, placeholder='…'), 30))
    ax.set_title(f"{tag} · {d['case_name']} vs {d['reference_name']}\n{name}", loc='left', fontsize=5.6,
                 color=A_COLOR if side == 'A only' else B_COLOR, pad=8)
    ax.text(0, -.22, '\n'.join(textwrap.wrap(reason(ds, side, pathway), 52)), transform=ax.transAxes, fontsize=4.8,
            color=INK, va='top')
    strips = ((d['log_ratio'].to_numpy()[idx], DIVERGING, -2, 2, 'offset'), (score[idx], SCORE_MAP, -1, 2.5, 'score'),
              (selected[idx].astype(float), SELECT_MAP, 0, 1, 'B'))
    for j, (values, cmap, lo, hi, label) in enumerate(strips):
        strip = fig.add_subplot(cell[j + 1])
        strip.imshow(values[:, None], aspect='auto', cmap=cmap, vmin=lo, vmax=hi, interpolation='nearest')
        strip.set_xticks([])
        strip.set_yticks([])
        strip.set_xlabel(label, fontsize=4.8, rotation=90, labelpad=2, va='top')
        for spine in strip.spines.values():
            spine.set_visible(False)
for x0, (norm_, label) in zip((.12, .56), ((Normalize(-1, 1), 'shape: Δ(s) − mean Δ (natural log, clipped ±1)'),
                                           (Normalize(-2, 2), 'offset strip: mean Δ (natural log, clipped ±2)'))):
    cax = fig.add_axes([x0, .03, .3, .008])
    bar = fig.colorbar(plt.cm.ScalarMappable(norm=norm_, cmap=DIVERGING), cax=cax, orientation='horizontal')
    bar.set_label(label + ';  red = case higher', fontsize=5.2, labelpad=1)
    bar.ax.tick_params(labelsize=5)
save(fig, 'fig3_worked_examples')
plt.show()

## 6 · Summary

**The setting.** Both methods ran on one NB gene model, one grid, one gene universe and one set of
pathways (10–300 genes). Only the pathway method differs. Every count below is for the condition
split; "relabeled" gives the two balanced relabelings (method B: with the fair single-split null).

**Method tables** (calls; relabeled calls; NCDR; calibration share):

| Method | Human vs mouse (1,513 pathways) | AKI vs control (1,429 pathways) |
|---|---|---|
| A · cameraPR (primary) | 98; 46, 29; 0.38; 0.121 · **fails both** | 144; 87, 86; 0.60; 0.170 · **fails both** |
| A · cameraPR, use.ranks | 81; 28, 6; 0.21; 0.108 · fails calibration | 140; 69, 60; 0.46; 0.161 · fails both |
| A · GSEA, rank-normal | 146; 75, 80; 0.53; 0.147 · fails both | 203; 120, 146; 0.66; 0.212 · fails both |
| A · GSEA, raw `T_spatial` | 0; 51, 35; —; 0.176 · fails | 0; 46, 5; —; 0.207 · fails |
| Reference · our NB joint test | 32; 3, 0; 0.05; 0.067 · passes | 12; 0, 0; 0; 0.072 · passes |
| B · his ORA (default null) | 167 (72 redundancy clusters); 0, 0 | 464 (262 clusters); 2, 0 |
| B · his ORA, single-split null | 162 and 159; 0, 0 | 469 and 482; 2, 0 |
| B · set-curve calls (level) | 142; 1, 0 | 272; 42, 0 |

The reference joint test's VIF is fitted on the same relabelings that judge it. Fitted on one
relabeling and judged on the other, its share is 0.067 (human vs mouse) and 0.075 (AKI).

**Observations.**
1. **cameraPR on `T_spatial` is not specific in either dataset.** Under relabeling it calls 29–87
   pathways, and 12–17% of its relabeled p-values are ≤ 0.05 (limit 7.5%). GSEA is worse, as
   expected. The rank version of cameraPR passes the relabeling rule only for human vs mouse.
2. **The cause is not the 0.01 correlation setting (section 5.2, post hoc).** cameraPR's relabeled z
   are inflated by a constant factor, a = 1.51 (human vs mouse) and 2.13 (AKI), with an implied
   ρ ≈ 0. Raising `inter.gene.cor` therefore cannot fix it.
   - Part of the excess follows expression: relabeled `T_spatial` correlates with mean expression
     (Spearman 0.17–0.22).
   - Ranking within the joint test's 27 expression × detection × coverage strata cuts human vs
     mouse to 70 calls (relabeled 9, 0; share 0.082).
   - It does not rescue AKI (112; 62, 39; share 0.135). There, specimen deviations are coherent
     within pathways beyond expression. Our matched joint test sees the same inflation (a = 1.55)
     and absorbs it only because its VIF is fitted on the relabelings.
3. **Caleb's pipeline is specific under the fair null.** Its relabelings select 15–134 genes and call
   0–2 pathways. With his default null a relabeled run calls 0 by construction, because that null
   contains the condition split. His calls carry no p-value. The list-based calibration share
   (0.003–0.005) passes only because almost no gene is selected under relabeling.
4. **The two methods find mostly different pathways.** Over redundancy clusters, cameraPR and his
   ORA share 22 in human vs mouse (70 cameraPR-only, 50 ORA-only; Jaccard 0.15) and 50 in AKI
   (80 and 212; Jaccard 0.15). Our joint test overlaps his ORA even less (Jaccard 0.12 and 0.04).
5. **What each sees.**
   - **cameraPR calls pathways whose genes change shape strongly but in different ways.** Examples
     are branched-chain amino acid degradation, arginine/proline and glutathione metabolism. In the
     four worked examples, his selection keeps 88–100% of the genes but spreads them over 7–14 of
     his gene groups, so no single list is enriched.
   - **His ORA calls pathways whose genes share one direction**, as an offset or a broad gradient.
     In human vs mouse, one shape cluster (D4, 157 genes lower in human) carries 126 of his 167
     calls. It holds 8 proteasome subunits, which sit in many Reactome sets; the 126 calls form 41
     redundancy clusters. Oxidative phosphorylation is called through "Down · uniform", a constant
     offset that `T_spatial` removes on purpose (cameraPR FDR 0.56).
   - **In AKI, his ORA finds proliferation that cameraPR misses.** G2-M checkpoint, cell cycle and
     p53 come through "Up · gradient toward S3" (q 10⁻¹²–10⁻⁷). Their genes score only +0.15 to
     +0.40 on `T_spatial`, and their cameraPR FDR is 0.11–0.62. 195 of his 464 AKI calls come only
     through lists without a confirmed shape (uniform groups, cluster U0).
6. **Genes absent in one group do not drive either pathway list.** In human vs mouse, his selected
   genes are 10.6% nearly absent (base rate 8.4%; `T_spatial` top 10%: 0.5%). Yet his ORA calls have
   fewer nearly absent members (median share 1.8%) than all pathways (3.4%). His level-based
   set-curve calls have more (17.6%).

**Deviations from his defaults** (table in section 4.1).
- The NB curves replace his quasi-Poisson P-spline and his binning.
- The grid is 101 points over the four specimens' common range.
- The gene sets are ours, with the size cap raised from 150 to 300.
- The prior is off (`prior_k = 0`).
- The zones come from his own marker rule, computed once on the control mice and fixed for all runs.
- The node spacing is 1/29 for both coordinates.
- The expressed-elsewhere flags are not computed; no call uses them.
- **Not standardizable: the typical-gene reference.** His API does not expose it, so it stays on.
  The shift it removes spans 0.21 (human vs mouse) and 0.05 (AKI) on the natural-log scale.
- **Added because the prior is off: a floor of half a count per specimen.** Without it, a gene with
  no counts in a specimen has an infinite log ratio. The floor binds for 913 human vs mouse genes,
  nearly all absent in one species, and for 20 AKI genes.

**Runtime.** The shared gene model took 119 s (human vs mouse) and 134 s (AKI) for three partitions
on 36 workers. The pathway step for the condition split and two relabelings then took:
- cameraPR, about 1 s;
- GSEA, 8–10 s;
- our joint test, 15–20 s;
- his stages 06–07, 18–21 s.

**What this cannot tell you.** Relabelings mix the two conditions, so they cannot see an artefact
that follows the condition. With two specimens per group every result is descriptive. Section 5.2
was added after the method tables were seen.

## 7 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json')):
    meta = json.loads(sidecar.read_text())
    if meta['stage'] in SECONDS:
        cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                              'compute seconds': round(SECONDS[meta['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'datasets': INPUT_TABLE.to_dict(), 'n_jobs': N_JOBS,
    'gene_model': {'module': 'pseudospace.pathway_pipelines.standard_gene_model', 'basis_df': BASIS_DF,
                   'grid_points': GRID_POINTS, 'nb_gam': nb_gam.GENE_MODEL,
                   'seconds': GENE_MODEL_SECONDS},
    'method_a': {'statistic': 'T_spatial (NB LR), rank-normal scores', 'cameraPR': {'inter.gene.cor': CAMERA_COR, 'use.ranks': False},
                 'gsea': {'permutations': GSEA_PERMUTATIONS, 'seed': GSEA_SEED, 'weight': 1},
                 'reference_joint': {'n_null': N_NULL, 'seed': SEED, 'vif_fit': JOINT_FIT}},
    'method_b': {'package': EXTERNAL, 'compare_params': repr(COMPARE_PARAMS), 'pathway_params': repr(PATHWAY_PARAMS),
                 'node_spacing': NODE_SPACING, 'floor_counts': FLOOR_COUNTS, 'zone_markers': ZONE_MARKERS,
                 'zones': {ds: ZONES[ds].tolist() for ds in DATASETS},
                 'standardization': STANDARDIZATION.to_dict(orient='records'), 'runs': RUN_TABLE.to_dict(orient='records')},
    'redundancy_rule': {'jaccard_cut': REDUNDANCY_JACCARD, 'function': 'pseudospace_reconstruction.compare.pathways.cluster_gene_sets'},
    'methods': METHOD_TABLE.to_dict(orient='records'), 'overlap': OVERLAP.to_dict(orient='records'),
    'composition': COMPOSITION.to_dict(orient='records'), 'gene_lists': GENE_COMPOSITION.to_dict(orient='records'),
    'worked_examples': [{'dataset': ds, 'side': side, 'pathway': p, 'reason': reason(ds, side, p)} for ds, side, p in EXAMPLES],
    'guards': guard_log,
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'gseapy', 'rpy2', 'joblib', 'matplotlib',
                                            'pseudospace-reconstruction')}},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")